#  CrowdHuman Object Detection Benchmark & Analysis
This notebook covers the full pipeline for training and evaluating YOLO and RT-DETR models on the highly dense CrowdHuman dataset. It includes data exploration, ODGT-to-YOLO conversion, automated benchmarking, and visual error analysis.

##  Table of Contents
1. [Environment Setup & Imports](#part-1)
2. [Dataset Directory Setup](#part-2)
3. [Inspect Raw Annotations](#part-3)
4. [Define YOLO Conversion Function](#part-4)
5. [Execute YOLO Conversion](#part-5)
6. [Sanity Check Converted Files](#part-6)
7. [Visualize Ground Truth](#part-7)
8. [Benchmarking Configuration & VRAM Utilities](#part-8)
9. [Benchmarking Model Utilities](#part-9)
10. [Core Benchmark Function](#part-10)
11. [Execute Benchmark Suite & Save Results](#part-11)
12. [Visual Error Analysis - Helper Functions](#part-12)
13. [Error Analysis - Prepare Data & Load Models](#part-13)
14. [Execute Visual Error Analysis](#part-14)
15. [Error Analysis Summary & Visualization](#part-15)

# Part 1: Environment Setup & Imports
Install necessary packages, import libraries, and verify GPU availability.

In [ ]:
!pip install -q ultralytics

import os
import json
import shutil
import random
import gc
import re
import io
import glob
import traceback
from contextlib import redirect_stdout

import torch
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO, RTDETR

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Libraries imported successfully.")

# Part 2: Dataset Directory Setup
Define base paths and verify the folder structure of the CrowdHuman dataset.

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/loctran0941/crowdhuman"
BASE_PATH = os.path.join(DATASET_PATH, "CrowdHuman")

# Walk through dataset directories (up to 2 levels deep)
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    if level > 2:
        continue
    print("\n" + root)
    for file in files[:5]: # Showing first 5 for brevity
        print("   ", file)
    if len(files) > 5:
        print("    ...")

train_path = os.path.join(BASE_PATH, "train")
val_path = os.path.join(BASE_PATH, "val")
train_images = os.path.join(train_path, "images")
val_images = os.path.join(val_path, "images")

train_files = os.listdir(train_images)
val_files = os.listdir(val_images)

print("\nNumber of train images:", len(train_files))
print("Number of validation images:", len(val_files))

# Part 3: Inspect Raw Annotations
Read the original ODGT (JSON-line) annotation format to understand its structure before conversion.

In [ ]:
annotation_path = os.path.join(BASE_PATH, "annotation_val.odgt")

with open(annotation_path, "r") as f:
    first_line = f.readline()

data = json.loads(first_line)
print("Annotation Keys:", data.keys())
print("Image ID:", data["ID"])
print("Number of annotations in this image:", len(data["gtboxes"]))
print("\nFirst annotation sample:")
print(json.dumps(data["gtboxes"][0], indent=4))

# Part 4: Define YOLO Conversion Function
YOLO models require `.txt` files with normalized coordinates. We define a function to parse ODGT files, extract only the `person` class, normalize the bounding boxes, and save them.

In [ ]:
OUTPUT_PATH = "/kaggle/working/crowdhuman_yolo"

# Create required YOLO directories
directories = [
    os.path.join(OUTPUT_PATH, "images", "train"),
    os.path.join(OUTPUT_PATH, "images", "val"),
    os.path.join(OUTPUT_PATH, "labels", "train"),
    os.path.join(OUTPUT_PATH, "labels", "val"),
]
for directory in directories: 
    os.makedirs(directory, exist_ok=True)

def convert_crowdhuman(annotation_file, image_dir, output_image_dir, output_label_dir):
    converted, skipped, total_people = 0, 0, 0
    
    with open(annotation_file, "r") as f:
        for line in tqdm(f, desc=f"Converting {os.path.basename(annotation_file)}"):
            data = json.loads(line)
            image_id = data["ID"]
            image_name = image_id + ".jpg"
            source_image = os.path.join(image_dir, image_name)
            
            if not os.path.exists(source_image):
                skipped += 1
                continue

            with Image.open(source_image) as image:
                image_width, image_height = image.size

            yolo_labels = []
            for box in data["gtboxes"]:
                if box.get("tag") != "person": continue
                fbox = box.get("fbox")
                if fbox is None: continue
                x, y, width, height = fbox
                if width <= 0 or height <= 0: continue
                if box.get("extra", {}).get("ignore", 0) == 1: continue

                # Normalize coordinates for YOLO (center_x, center_y, width, height)
                center_x = max(0.0, min(1.0, (x + width / 2) / image_width))
                center_y = max(0.0, min(1.0, (y + height / 2) / image_height))
                w_norm = max(0.0, min(1.0, width / image_width))
                h_norm = max(0.0, min(1.0, height / image_height))

                yolo_labels.append(f"0 {center_x:.6f} {center_y:.6f} {w_norm:.6f} {h_norm:.6f}")
                total_people += 1

            label_path = os.path.join(output_label_dir, image_id + ".txt")
            with open(label_path, "w") as label_file:
                label_file.write("\n".join(yolo_labels))

            output_image = os.path.join(output_image_dir, image_name)
            if not os.path.exists(output_image):
                # OPTIMIZATION: Create a symbolic link instead of copying the file
                os.symlink(source_image, output_image)
            converted += 1

    return converted, skipped, total_people

# Part 5: Execute YOLO Conversion
Run the conversion function on both train and validation splits, and generate the required dataset `.yaml` file.

In [ ]:
TRAIN_ANNOTATION = os.path.join(BASE_PATH, "annotation_train.odgt")
VAL_ANNOTATION = os.path.join(BASE_PATH, "annotation_val.odgt")

# Convert Train Set
c_tr, s_tr, p_tr = convert_crowdhuman(
    TRAIN_ANNOTATION, train_images, 
    os.path.join(OUTPUT_PATH, "images", "train"), 
    os.path.join(OUTPUT_PATH, "labels", "train")
)
print(f"Train - Converted: {c_tr}, Skipped: {s_tr}, Persons: {p_tr}")

# Convert Val Set
c_val, s_val, p_val = convert_crowdhuman(
    VAL_ANNOTATION, val_images, 
    os.path.join(OUTPUT_PATH, "images", "val"), 
    os.path.join(OUTPUT_PATH, "labels", "val")
)
print(f"Val - Converted: {c_val}, Skipped: {s_val}, Persons: {p_val}")

# Generate dataset YAML file
yaml_content = f"""path: {OUTPUT_PATH}
train: images/train
val: images/val
names:
  0: person"""

yaml_path = os.path.join(OUTPUT_PATH, "crowdhuman.yaml")
with open(yaml_path, "w") as f: 
    f.write(yaml_content)

print(f"\nDataset YAML saved to: {yaml_path}")

# Part 6: Sanity Check Converted Files
Verify that the number of images and label text files match in the new YOLO dataset folder.

In [ ]:
out_train_images = os.path.join(OUTPUT_PATH, "images", "train")
out_train_labels = os.path.join(OUTPUT_PATH, "labels", "train")
out_val_images = os.path.join(OUTPUT_PATH, "images", "val")
out_val_labels = os.path.join(OUTPUT_PATH, "labels", "val")

print("TRAIN SET")
print(f"Images: {len(os.listdir(out_train_images))} | Labels: {len(os.listdir(out_train_labels))}")

print("\nVALIDATION SET")
print(f"Images: {len(os.listdir(out_val_images))} | Labels: {len(os.listdir(out_val_labels))}")

# Part 7: Visualize Ground Truth
Pick a random image from the converted validation set and plot the normalized YOLO bounding boxes back onto the image in pixel coordinates.

In [ ]:
image_files = os.listdir(out_val_images)
image_name = random.choice(image_files)
image_path = os.path.join(out_val_images, image_name)
label_path = os.path.join(out_val_labels, image_name.replace(".jpg", ".txt"))

image = Image.open(image_path)
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)
image_width, image_height = image.size

with open(label_path, "r") as f:
    for line in f:
        values = line.strip().split()
        if len(values) != 5: continue
            
        class_id, cx, cy, w, h = map(float, values)
        
        # Convert YOLO normalized back to pixels
        box_width = w * image_width
        box_height = h * image_height
        x = (cx * image_width) - box_width / 2
        y = (cy * image_height) - box_height / 2

        rect = patches.Rectangle((x, y), box_width, box_height, linewidth=1.5, fill=False, edgecolor="lime")
        ax.add_patch(rect)

ax.set_title(f"CrowdHuman Ground Truth - {image_name}")
ax.axis("off")
plt.show()

# Part 8: Benchmarking Configuration & VRAM Utilities
Set up the parameters for the benchmark suite and define helper functions to clean and monitor GPU memory. 

## Hyperparameter Configuration
| Hyperparameter | Value | Pipeline | Description & Impact |
| :--- | :---: | :--- | :--- |
| **`IMG_SIZE`** (`imgsz`) | `640` | Benchmark & Analysis | The input resolution images are resized to. Higher values improve small object detection but increase VRAM usage. |
| **`BATCH_SIZE`** (`batch`) | `8` | Benchmark | Number of images processed simultaneously. Reduce if you get CUDA Out-Of-Memory (OOM) errors. |
| **`MAX_DET`** (`max_det`) | `310` | Benchmark & Analysis | Maximum bounding boxes output per image. Set high for CrowdHuman to ensure no pedestrians are missed. |
| **`CONF_THRESHOLD`** (`conf`) | `0.25` | Analysis | Minimum confidence score. Lowering it increases recall but decreases precision (more false positives). |
| **`IOU_THRESHOLD`** (`iou`) | `0.50` | Analysis | Intersection over Union threshold used for Non-Maximum Suppression to remove duplicate boxes. |
| **`WORKERS`** (`workers`) | `0` | Benchmark | CPU threads used for data loading. Set to `0` to prevent multiprocessing hanging issues in notebooks. |
| **`DEVICE`** (`device`) | `0` | Benchmark & Analysis | The index of the GPU used for processing (`cuda:0`). |

In [ ]:
DATA_YAML = "/kaggle/working/crowdhuman_yolo/crowdhuman.yaml"
IMG_SIZE, BATCH_SIZE, MAX_DET = 640, 8, 310
DEVICE, DEVICE_STR, WORKERS = 0, "cuda:0", 0
BENCHMARK_OUTPUT_DIR = "/kaggle/working/crowdhuman_model_benchmark"
os.makedirs(BENCHMARK_OUTPUT_DIR, exist_ok=True)

MODELS_DICT = {
    "YOLO26n": "yolo26n.pt", "YOLO26s": "yolo26s.pt", "YOLO26m": "yolo26m.pt",
    "YOLO12n": "yolo12n.pt", "YOLO12s": "yolo12s.pt", "YOLO12m": "yolo12m.pt", "YOLO12l": "yolo12l.pt",
    "YOLO11n": "yolo11n.pt", "YOLO11s": "yolo11s.pt", "YOLO11m": "yolo11m.pt",
    "YOLOv8n": "yolov8n.pt", "YOLOv8s": "yolov8s.pt", "YOLOv8m": "yolov8m.pt",
    "RT-DETR-l": "rtdetr-l.pt", "RT-DETR-x": "rtdetr-x.pt",
}
benchmark_results = []

def gb(value): 
    return value / (1024 ** 3)

def clean_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try: torch.cuda.ipc_collect()
        except: pass

def get_gpu_memory():
    if not torch.cuda.is_available(): return 0.0, 0.0
    return gb(torch.cuda.memory_allocated(DEVICE)), gb(torch.cuda.memory_reserved(DEVICE))

# Part 9: Benchmarking Model Utilities
Helper functions to safely push models (especially complex ones like RT-DETR) to the GPU and accurately extract parameter counts and GFLOPs.

In [ ]:
def move_model_to_device(model):
    try: model.to(DEVICE_STR)
    except Exception as e: print(f"WARNING: model.to failed:\n{e}")
    try:
        for module in model.model.modules():
            for name, buffer in module.named_buffers(recurse=False):
                if buffer is not None and torch.is_tensor(buffer) and buffer.device.type != "cuda":
                    try: setattr(module, name, buffer.to(DEVICE_STR))
                    except: pass
    except: pass

def fix_detr_device(model):
    fixed = []
    try:
        for module in model.model.modules():
            for attr in ["valid_mask", "anchors"]:
                if hasattr(module, attr):
                    value = getattr(module, attr)
                    if torch.is_tensor(value) and value.device.type != "cuda":
                        try:
                            setattr(module, attr, value.to(DEVICE_STR))
                            fixed.append(attr)
                        except: pass
    except: pass

def get_model_stats(model, model_name):
    params_m, gflops = None, None
    try: params_m = sum(p.numel() for p in model.model.parameters()) / 1e6
    except: pass
    
    try:
        f = io.StringIO()
        with redirect_stdout(f): model.info(verbose=True)
        raw_text = f.getvalue()
        clean_text = re.sub(r'\x1B\[[0-9;]*[mK]', '', raw_text)
        matches = re.findall(r"([0-9]+(?:\.[0-9]+)?)\s*GFLOPs", clean_text, flags=re.IGNORECASE)
        if matches: gflops = float(matches[-1])
    except: pass
    
    return params_m, gflops

# Part 10: Core Benchmark Function
This function takes a single model, loads it, runs validation on the CrowdHuman dataset, collects all metrics, and safely unloads it from memory.

In [ ]:
def run_single_benchmark(model_name, model_path):
    print(f"\n{'=' * 80}\nBENCHMARKING {model_name}\n{'=' * 80}")
    model, results = None, None
    clean_gpu()
    
    if torch.cuda.is_available():
        try: torch.cuda.reset_peak_memory_stats(DEVICE)
        except: pass
        
    base_allocated, base_reserved = get_gpu_memory()
    
    try:
        print(f"Loading {model_path}...")
        model = RTDETR(model_path) if "DETR" in model_name.upper() else YOLO(model_path)
        move_model_to_device(model)
        if "DETR" in model_name.upper(): fix_detr_device(model)
    except Exception as e:
        print(f"FAILED TO LOAD {model_name}: {e}")
        benchmark_results.append({"Model": model_name, "Status": "FAILED", "Error": str(e)})
        return

    params_m, gflops = get_model_stats(model, model_name)
    
    try:
        print(f"Starting validation for {model_name}...")
        results = model.val(data=DATA_YAML, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE, 
                            workers=WORKERS, max_det=MAX_DET, plots=False, verbose=False, split="val")
        
        precision, recall, map50, map5095 = float(results.box.mp), float(results.box.mr), float(results.box.map50), float(results.box.map)
        inference_ms = float(results.speed["inference"])
        fps = 1000.0 / inference_ms if inference_ms > 0 else None
        
        peak_alloc = gb(torch.cuda.max_memory_allocated(DEVICE)) if torch.cuda.is_available() else None
        
        benchmark_results.append({
            "Model": model_name, "Status": "SUCCESS", "Parameters_M": params_m, "GFLOPs": gflops, 
            "Precision": precision, "Recall": recall, "mAP50": map50, "mAP50-95": map5095, 
            "GPU_Inference_ms": inference_ms, "GPU_FPS": fps, "Peak_GPU_GB": peak_alloc
        })
        print(f"✓ {model_name} completed. mAP50: {map50:.4f}, FPS: {fps:.1f}")
        
    except Exception as e:
        print(f"VALIDATION FAILED: {model_name}\n{e}")
        benchmark_results.append({"Model": model_name, "Status": "FAILED", "Error": str(e)})

    del model, results
    clean_gpu()

# Part 11: Execute Benchmark Suite & Save Results
Iterate through all models in the dictionary, run the benchmark function, and export the results to CSV files, displaying a pandas DataFrame.

In [ ]:
# Run loop
for model_name, model_path in MODELS_DICT.items():
    run_single_benchmark(model_name, model_path)

# Aggregate results
df_bench = pd.DataFrame(benchmark_results)

if not df_bench.empty:
    # Sort by mAP
    if "mAP50-95" in df_bench.columns:
        df_bench["_success"] = (df_bench["Status"] == "SUCCESS")
        df_bench = df_bench.sort_values(by=["_success", "mAP50-95"], ascending=[False, False]).drop(columns=["_success"]).reset_index(drop=True)

    csv_path = os.path.join(BENCHMARK_OUTPUT_DIR, "crowdhuman_benchmark.csv")
    df_bench.to_csv(csv_path, index=False)
    
    print(f"\n{'=' * 100}\nFINAL BENCHMARK RESULTS\n{'=' * 100}")
    # Show successful models formatted nicely
    success_df = df_bench[df_bench["Status"] == "SUCCESS"].dropna(axis=1, how='all')
    if not success_df.empty:
        display(success_df.round(4))
    else:
        print("No models completed successfully.")

# Part 12: Visual Error Analysis - Helper Functions
Define logic to calculate Intersection over Union (IoU), match Ground Truth (GT) boxes to Predictions, and draw bounding boxes for side-by-side visual analysis.

In [ ]:
ANALYSIS_OUTPUT_DIR = "/kaggle/working/crowdhuman_error_analysis"
os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

NUM_IMAGES_TO_ANALYZE, CONF_THRESHOLD, IOU_THRESHOLD = 5, 0.25, 0.50

def load_ground_truth(image_path, labels_dir):
    label_path = os.path.join(labels_dir, os.path.splitext(os.path.basename(image_path))[0] + ".txt")
    image = cv2.imread(image_path)
    if image is None or not os.path.exists(label_path): return []
    h, w = image.shape[:2]
    boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid, xc, yc, bw, bh = map(float, parts[:5])
            x1, y1 = max(0, min(w - 1, (xc - bw / 2) * w)), max(0, min(h - 1, (yc - bh / 2) * h))
            x2, y2 = max(0, min(w - 1, (xc + bw / 2) * w)), max(0, min(h - 1, (yc + bh / 2) * h))
            boxes.append({"class_id": int(cid), "box": [x1, y1, x2, y2]})
    return boxes

def calculate_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])) + (max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])) - inter
    return inter / union if union > 0 else 0.0

def match_boxes(ground_truth, predictions, iou_threshold=0.5):
    matches, used_gt, used_preds = [], set(), set()
    candidates = sorted([(calculate_iou(p["box"], gt["box"]), p_idx, gt_idx) 
                         for p_idx, p in enumerate(predictions) 
                         for gt_idx, gt in enumerate(ground_truth) 
                         if calculate_iou(p["box"], gt["box"]) >= iou_threshold], reverse=True)
    for iou, p_idx, gt_idx in candidates:
        if p_idx not in used_preds and gt_idx not in used_gt:
            used_preds.add(p_idx); used_gt.add(gt_idx)
            matches.append({"prediction": p_idx, "ground_truth": gt_idx, "iou": iou})
    fp = [i for i in range(len(predictions)) if i not in used_preds]
    fn = [i for i in range(len(ground_truth)) if i not in used_gt]
    return matches, fp, fn

def draw_box(image, box, color, label=None):
    x1, y1, x2, y2 = map(int, box)
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    if label:
        (tw, th), bl = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        ty = max(y1, th + 5)
        cv2.rectangle(image, (x1, ty - th - 5), (x1 + tw + 4, ty + bl - 2), color, -1)
        cv2.putText(image, label, (x1 + 2, ty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

# Part 13: Error Analysis - Prepare Data & Load Models
Select a subset of validation images and load the models into memory for the deep dive error analysis.

In [ ]:
extensions = ["*.jpg", "*.jpeg", "*.png"]
all_val_imgs = sorted([f for ext in extensions for f in glob.glob(os.path.join(out_val_images, ext))])

if not all_val_imgs: 
    raise RuntimeError("No validation images found.")

indices = np.linspace(0, len(all_val_imgs) - 1, min(NUM_IMAGES_TO_ANALYZE, len(all_val_imgs)), dtype=int)
selected_images = [all_val_imgs[i] for i in indices]

# Load models into RAM/VRAM
loaded_models = {}
for name, path in MODELS_DICT.items():
    print(f"Loading {name} for analysis...")
    try:
        model = YOLO(path)
        model.to(DEVICE_STR)
        loaded_models[name] = model
    except Exception as e: 
        print(f"✗ Could not load {name}")

# Part 14: Execute Visual Error Analysis
Run inferences on the selected sample images, compute True Positives, False Positives (FP), and False Negatives (FN). Draw GT on the left and Predictions on the right.

In [ ]:
analysis_results = []

for model_name, model in loaded_models.items():
    print(f"\nAnalyzing: {model_name}")
    model_out_dir = os.path.join(ANALYSIS_OUTPUT_DIR, model_name)
    os.makedirs(model_out_dir, exist_ok=True)

    for i, img_path in enumerate(selected_images):
        img_name = os.path.splitext(os.path.basename(img_path))[0]
        image = cv2.imread(img_path)
        
        ground_truth = load_ground_truth(img_path, out_val_labels)
        
        # Inference
        results = model.predict(source=image, imgsz=IMG_SIZE, conf=CONF_THRESHOLD, 
                                iou=IOU_THRESHOLD, max_det=MAX_DET, device=DEVICE, verbose=False)[0]
        
        predictions = []
        if results.boxes is not None:
            boxes, confs, cls = results.boxes.xyxy.cpu().numpy(), results.boxes.conf.cpu().numpy(), results.boxes.cls.cpu().numpy()
            predictions = [{"box": boxes[j].tolist(), "confidence": float(confs[j]), "class_id": int(cls[j])} for j in range(len(boxes))]

        matches, fp_list, fn_list = match_boxes(ground_truth, predictions, IOU_THRESHOLD)
        tp, fp, fn = len(matches), len(fp_list), len(fn_list)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        # Draw Side-by-Side
        viz_gt, viz_pred = image.copy(), image.copy()
        cv2.putText(viz_gt, "GROUND TRUTH", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
        cv2.putText(viz_pred, f"PREDICTIONS ({model_name})", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)

        for gt_idx, gt in enumerate(ground_truth):
            matched = any(m["ground_truth"] == gt_idx for m in matches)
            draw_box(viz_gt, gt["box"], (0, 255, 0), f"GT" if matched else f"FN")
        
        for pred_idx, pred in enumerate(predictions):
            matched = next((m for m in matches if m["prediction"] == pred_idx), None)
            if matched: draw_box(viz_pred, pred["box"], (255, 0, 0), f"{pred['confidence']:.2f}")
            else: draw_box(viz_pred, pred["box"], (0, 0, 255), f"FP {pred['confidence']:.2f}")

        viz_combined = np.hstack((viz_gt, viz_pred))
        
        # Add stats text
        summary_text = [f"TP: {tp}", f"FP: {fp}", f"FN: {fn}", f"Prec: {precision:.2f}", f"Rec: {recall:.2f}"]
        for j, text in enumerate(summary_text):
            cv2.putText(viz_combined, text, (20, 85 + 30 * j), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out_path = os.path.join(model_out_dir, f"{img_name}_analysis.jpg")
        cv2.imwrite(out_path, viz_combined)
        
        analysis_results.append({
            "Model": model_name, "Image": img_name, "TP": tp, "FP": fp, "FN": fn, 
            "Precision": precision, "Recall": recall
        })

# Part 15: Error Analysis Summary & Visualization
Aggregate the object-level statistics (TP, FP, FN) into a summary DataFrame, and display one of the saved side-by-side evaluation images in the notebook.

In [ ]:
df_analysis = pd.DataFrame(analysis_results)

if not df_analysis.empty:
    model_summary = df_analysis.groupby("Model").agg({
        "TP": "sum", "FP": "sum", "FN": "sum", 
        "Precision": "mean", "Recall": "mean"
    }).reset_index()
    
    print("\nOVERALL ERROR ANALYSIS SUMMARY")
    display(model_summary.round(3))

print(f"\n{'=' * 80}\nSAMPLE VISUALIZATION\n{'=' * 80}")
for model_name in loaded_models.keys():
    files = sorted(glob.glob(os.path.join(ANALYSIS_OUTPUT_DIR, model_name, "*.jpg")))
    if files:
        plt.figure(figsize=(24, 10)) 
        plt.imshow(cv2.cvtColor(cv2.imread(files[0]), cv2.COLOR_BGR2RGB))
        plt.title(f"{model_name} - Ground Truth vs Predictions")
        plt.axis("off")
        plt.show()
        break # Just show one to avoid flooding the notebook